In [51]:
from crewai import Agent,Task,Crew,Process,LLM
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient
from typing import List
from crewai.tools import tool
import agentops
import os 

In [52]:
load_dotenv()

llm = LLM(
    model="command-a-03-2025",
    api_key=os.getenv("COHERE_API_KEY"),
    base_url="https://api.cohere.com/compatibility/v1",
    temperature=0
)

In [53]:
session = agentops.init(
    api_key=os.getenv("AGENTOPS_API_KEY"),
    #this code to prevent close after first agent 
    skip_auto_end_session=True
)
print(session)

None


In [54]:
search_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [55]:
outpur_dir = "./ai-agent-output"
os.makedirs(outpur_dir,exist_ok=True)

### FIRST AGENT & TASK

In [56]:
no_keywords = 10
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(...,title="suggested search quieries to be passed to the search engine")

#every agent can make one task or multiple tasks
#each agent has role, goal, backstory,llm  
Search_Quaries_Recommendition_Agent = Agent(
    role="Search_Quaries_Recommendition_Agent",
    goal="\n".join([
        "to provide alist of suggested search queries to be passed to the search engine",
        "the queries must be varied and looking for specific items"
           ]),
    backstory="You are an expert search query strategist specializing in product research and online shopping. You analyze user requests and transform them into clear, specific, and diverse search queries. Your goal is to cover different brands, models, specifications, price ranges, and purchasing options to help search engines find the most relevant products and deals."    ,
    llm = llm,
    verbos=True
)
#each task has description , expected output and agent to do this task , expected output , output json
# optional can take async,output file 
Search_Quaries_Recommendition_Task = Task(
    description="\n".join([
        "Generate exactly {no_keyword} highly relevant and diverse search keywords for finding {product_name}.",

        "The keywords will be passed to a separate search engine agent that will perform the actual web search.",

        "The product must be available for purchase and deliverable in {country_name}.",

        "The search will be restricted to these e-commerce websites:",
        "{website_list}",

        "IMPORTANT: Do NOT include website names, domains, URLs, or 'site:' operators in the keywords.",

        "Do NOT generate URLs or links.",

        "Do NOT return product pages or search results.",

        "Do NOT include blog posts, articles, reviews, forums, news websites, or informational content.",

        "Generate keywords that describe the product and its purchasing requirements.",

        "Vary the keywords using different brands, models, specifications, features, price ranges, and product variations.",

        "Each keyword should be a concise search phrase that can be directly passed to a search engine.",

        "Return ONLY the list of keywords.",
    ]),
    #json to prevent words in introduction and conclusion
    expected_output="A JSON containing alist of suggested search queries",
    #we will create pydantic scheme for output json 
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(outpur_dir,"step1.json"),
    agent=Search_Quaries_Recommendition_Agent
)

### SECOND AGENT

In [ ]:
#create nested pydantic
class SingleSearchResult(BaseModel):
    title: str
    url: str
    content: str
    score: float
    search_query: str
    
class AllSearchResult(BaseModel):
    result:List[SingleSearchResult]


#  we need google search tool
#will make custom tool (tavily)
# doc string is must in custom tool to tell what tool make
@tool
def search_engine_tool(query:str):
    """
    Search the web for relevant e-commerce product pages using the provided query.

    Args:
        query: A search query describing the product or information to find.
                The query should be specific and focused on purchasable products.

    Returns:
        Search results containing relevant web pages matching the query.
    """
    return search_client.search(query)


# take keywords and search google for products based on the suggested search query
Search_Engine_Agent = Agent(
    role = "Search Engine Agent",
    goal = "To search for products based on the suggessted search query",
    backstory="""
    You are an expert web search specialist focused on finding
    purchasable products across e-commerce websites.

    You receive carefully generated search queries from a Search Query
    Recommendation Agent and use them to perform accurate web searches.

    Your responsibility is to find relevant product pages, product listings,
    and online stores. You prioritize results that contain actual products
    available for purchase and ignore blogs, news articles, forums, and
    informational pages.

    You carefully follow the provided search queries and return the most
    relevant search results without modifying the user's search intent.
    """,
    llm = llm,
    verbose=True,
    tools=[search_engine_tool]

)
Search_Engine_Task = Task(
    description="/n".join([
        "The task is to search for products based on the suggested search queries.",
        "You have to collect results from multiple search queries.",
        "Ignore any susbicious links or not an ecommerce single product website link.",
        "Ignore any search results with confidence score less than ({score_th}) .",
        "The search results will be used to compare prices of products from different websites."
    ]),
    expected_output="A JSON object containing the search results",
    output_json=AllSearchResult,
    output_file=os.path.join(outpur_dir,"step2.json"),
    agent=Search_Engine_Agent
)

In [58]:
run = Crew(
    agents=[
        Search_Quaries_Recommendition_Agent,
        Search_Engine_Agent, 
            ],
    tasks=[
        Search_Quaries_Recommendition_Task,
        Search_Engine_Task
        ],
    process=Process.sequential
)

In [59]:
results =  await run.kickoff_async(
    inputs={
        "product_name": "coffee machine for the office",
        "website_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
        "country_name": "Egypt",
        "no_keyword": 10,
        "score_th":0.10
    }
    )

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Engine Agent                                                                                     │
│                                                                                                                 │
│  Task: The task is to search for products based on the suggested search queries./nYou have to collect results   │
│  from multiple search queries./nIgnore any susbicious links or not an ecommerce single product website          │
│  link./nIgnore any search results with confidence score less than (0.1) ./nThe search results will be used to   │
│  compare prices of products from different websites.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_engine_tool executed with result: {'query': 'Office coffee machines Egypt', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.fajtradingllc.com/pages/coffee-machines-in-egypt', 'title': 'Buy C...
Tool search_engine_tool executed with result: {'query': 'Commercial coffee makers for offices in Egypt', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://ecoffee1.com/best-coffee-machines-egypt-cafes-offices...
Tool search_engine_tool executed with result: {'query': 'Automatic espresso machines for office use Egypt', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://ecoffee1.com/best-espresso-nespresso-coffee-machin...
Tool search_engine_tool executed with result: {'query': 'Best coffee machines for large offices Egypt', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://ecoffee1.com/best-coffee-machines-egypt-cafes-offices-...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Engine Agent                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│      "result": [                                                                                                │
│          {                                                                                                      │
│              "titile": "Buy Coffee Machines in Egypt - Coffee Maker Best Offer",                                │
│              "url": "https://www.fajtradingllc.com/pages/coffee-machines-in-egypt",                             │
│              "content": "F A J Trading L.L.C is the best dealer and supplier of coffee machines designed to     │
│  help you brew the perfect cup at home, in the office, or at a restaurant. Whether you prefer rich espresso,    │
│  frothy cappuccino, or traditional Arabic coffee, you’ll find exactly what you need in Cairo, Elexendra, Suez,  │
│  Demitta, Sharkia, Port said, Kaliubia, Giza, Dakhalia, Shakira, Qaliubia, Ismailia, Borg el Arab, Menufia,     │
│  Behaira, Beni Suef Faiyum, Minya in Egypt.We offer a variety of machines, including [...] + All Navigation\n   │
│  + - Coffee Machines Navigation\n    - Professional Coffee Machines\n    - Automatic Coffee Machine\n    -      │
│  Office Coffee Machines\n    - Home Coffee Machines\n    - Capsule Coffee Machines\n    - Coffee Brewers\n      │
│  - Turkish Coffee Machines\n    - Coffee Machine Equipment\n    - Coffee Accessories\n  + - Home Appliances     │
│  Navigation\n    - Refrigerators\n    - Washing Machines\n    - Cooker\n    - Ovens\n    - Dishwashers\n    -   │
│  Kitchen Hoods\n    - Vacuum Cleaners\n    - Hobs [...] Coffee Equipments\n\nF A J Trading                      │
│  L.L.C\n\nBlenders\n\nF A J Trading L.L.C\n\nCoffee Tamper\n\nF A J Trading L.L.C\n\nTurkish Coffee             │
│  Machines\n\nBean to Cup Coffee & Espresso Machines\n\n## Bean to Cup Coffee & Espresso Machines\n\nUnlock      │
│  professional performance at home with Sage. Like a professional bean to cup coffee machine, ours uses the 4    │
│  Keys Formula to deliver delicious third-wave speciality coffee.\n\nShop Now\n\nF A J Trading L.L.C\n\n###      │
│  Boost productivity with cafe-quality office coffee.",                                                          │
│              "score": 0.8719229,                                                                                │
│              "search_query": "Office coffee machines Egypt"                                                     │
│          },                                                                                                     │
│          {                                                                                                      │
│              "titile": "13 Best Coffee Machines in Egypt for Cafés, Offices & Events",                          │
│              "url": "https://ecoffee1.com/best-coffee-machines-egypt-cafes-offices-events",                     │
│              "content": "Investing in a coffee machine for a café, office, or event setup in Egypt isn’t just   │
│  a practical purchase—it’s a decision that directly impacts customer satisfaction, brand perception, and        │
│  operational efficiency. Whether you’re running a trendy Zamalek café, managing an upscale corporate lounge in  │
│  New Cairo, or equipping a short-term exhibition booth 

ERROR:instructor.v2.retry:API call failed on attempt 1: Error code: 422 - {'error_type': 'NO_TOOL_CALL_OR_RESPONSE_GENERATED', 'id': '0d7ea8e4-e81e-4e21-a951-ef93a2002803', 'message': 'No tool calls or response was generated. Try updating messages or tool definitions'}
ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: Error code: 422 - {'error_type': 'NO_TOOL_CALL_OR_RESPONSE_GENERATED', 'id': '0d7ea8e4-e81e-4e21-a951-ef93a2002803', 'message': 'No tool calls or response was generated. Try updating messages or tool definitions'}


CancelledError: 

ERROR:instructor.v2.retry:API call failed on attempt 1: Error code: 422 - {'error_type': 'INVALID_TOOL_GENERATION', 'id': '4f7a2273-8024-4a28-bc6c-4001f302c42a', 'message': 'your request resulted in an invalid tool generation. Try updating the messages or tool definitions'}
ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: Error code: 422 - {'error_type': 'INVALID_TOOL_GENERATION', 'id': '4f7a2273-8024-4a28-bc6c-4001f302c42a', 'message': 'your request resulted in an invalid tool generation. Try updating the messages or tool definitions'}


In [ ]:
results

CrewOutput(raw='{"queries":["Office coffee machines Egypt delivery","Commercial coffee makers for offices in Egypt","Automatic espresso machines for office use Egypt","Best coffee machines for large offices Egypt","Affordable office coffee machines Egypt","Nespresso professional coffee machines Egypt","Bean-to-cup coffee machines for offices Egypt","High-capacity coffee makers for offices Egypt","Office coffee machines with milk frother Egypt","Energy-efficient office coffee machines Egypt"]}', pydantic=None, json_dict={'queries': ['Office coffee machines Egypt delivery', 'Commercial coffee makers for offices in Egypt', 'Automatic espresso machines for office use Egypt', 'Best coffee machines for large offices Egypt', 'Affordable office coffee machines Egypt', 'Nespresso professional coffee machines Egypt', 'Bean-to-cup coffee machines for offices Egypt', 'High-capacity coffee makers for offices Egypt', 'Office coffee machines with milk frother Egypt', 'Energy-efficient office coffee m